# FINAL PROJECT REPORT - PARALLEL PROGRAMMING (CSC14120)
## Title: Autoencoder-based Unsupervised Feature Learning for Image Classification

**Group Information:**
* **Member 1:** Le Dai Hoa - 22120108
* **Member 2:** Nguyen Tuong Bach Hy - 22120455
* **Member 3:** Le Hoang Vu - 22120461

**Link Video Demo:**

---
## Instructions for Google Colab:
1. Zip project folder (exclude data/)
2. Upload zip file when prompted
3. Run all cells in order

---
# Section 0: Environment Setup
---

In [ ]:
# 0.1 Check GPU
!nvidia-smi
!nvcc --version

In [ ]:
# 0.2 Upload and extract project
from google.colab import files
import zipfile, os

print("Upload project zip file:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

for root, dirs, _ in os.walk('project'):
    if 'src' in dirs:
        %cd {root}
        break

print("\nProject structure:")
!ls -la

In [ ]:
# 0.3 Download CIFAR-10 dataset
import urllib.request, tarfile
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 'data/cifar.tar.gz')
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/
    !rm -rf data/cifar-10-batches-bin data/cifar.tar.gz

print('CIFAR-10 files:')
!ls data/*.bin

---
# Section 1: Problem Description
---

## 1.1 Problem Statement

### Image Classification Task
Image classification is a fundamental computer vision task where the goal is to assign a label to an image from a predefined set of categories. Traditional approaches rely on **hand-crafted features** (SIFT, HOG, etc.), which require domain expertise and don't generalize well.

### Motivation for GPU Acceleration
Training neural networks involves massive parallel computations:
- **Convolution operations:** O(N × C_out × H × W × C_in × K²) per layer
- **50,000 training images × 20 epochs × multiple layers** = billions of operations

CPU training would take **~87 hours** for our autoencoder. GPU parallelization can reduce this to **< 10 minutes** (>500× speedup).

### Our Approach: Autoencoder-based Feature Learning
Instead of hand-crafted features, we use an **Autoencoder** to automatically learn meaningful representations:
- **Unsupervised learning:** No labels needed during feature learning
- **Encoder** compresses 32×32×3 images → 8,192-dim feature vectors
- **Decoder** reconstructs images from features (training only)
- Features are then used with **SVM** for classification

### Two-Stage Pipeline
```
Stage 1: Train Autoencoder (unsupervised)
   Input Image → Encoder → Latent (8192-dim) → Decoder → Reconstructed Image
                           ↓
Stage 2: Classification (supervised)
   Input Image → Trained Encoder → Features → SVM → Class Label
```

---
## 1.2 CIFAR-10 Dataset Overview

### Dataset Specifications
| Property | Value |
|:---------|:------|
| Image Size | 32 × 32 pixels (RGB) |
| Total Images | 60,000 |
| Training Set | 50,000 images (5,000 per class) |
| Test Set | 10,000 images (1,000 per class) |
| Number of Classes | 10 |
| Format | Binary files (uint8 pixel values) |

### Classes
`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`

### Data Preprocessing Steps
1. **Load binary files:** Parse format (1 byte label + 3072 bytes image per record)
2. **Reshape:** Convert flat array to (32, 32, 3) RGB format
3. **Normalize:** Convert uint8 [0, 255] → float32 [0, 1]
4. **Batch generation:** Create mini-batches of 64 images
5. **Shuffling:** Random shuffle at each epoch for better training

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CIFAR10_LABELS = ["airplane", "automobile", "bird", "cat", "deer",
                  "dog", "frog", "horse", "ship", "truck"]

def load_cifar10_batch(file_path):
    with open(file_path, 'rb') as f:
        data = np.frombuffer(f.read(), dtype=np.uint8)
    data = data.reshape(-1, 3073)
    labels = data[:, 0]
    images = data[:, 1:].reshape(-1, 3, 32, 32)
    images = np.transpose(images, (0, 2, 3, 1))
    return images, labels

# Load datasets
train_images, train_labels = [], []
for i in range(1, 6):
    imgs, lbls = load_cifar10_batch(f"data/data_batch_{i}.bin")
    train_images.append(imgs)
    train_labels.append(lbls)
train_images = np.concatenate(train_images)
train_labels = np.concatenate(train_labels)
test_images, test_labels = load_cifar10_batch("data/test_batch.bin")

print(f"Training set: {train_images.shape}")
print(f"Test set: {test_images.shape}")

# Show samples
plt.figure(figsize=(12, 3))
for i in range(10):
    idx = np.where(train_labels == i)[0][0]
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_images[idx])
    plt.title(CIFAR10_LABELS[i], fontsize=9)
    plt.axis("off")
plt.suptitle("CIFAR-10 Sample Images", fontsize=12)
plt.tight_layout()
plt.show()

---
## 1.3 Autoencoder Architecture

### Architecture Diagram
```
INPUT (32,32,3)
    ↓
[Conv2D 256 + ReLU] → (32,32,256)
    ↓
[MaxPool 2×2]       → (16,16,256)
    ↓
[Conv2D 128 + ReLU] → (16,16,128)
    ↓
[MaxPool 2×2]       → (8,8,128) ← LATENT SPACE (8,192 features)
    ↓
[Conv2D 128 + ReLU] → (8,8,128)
    ↓
[UpSample 2×2]      → (16,16,128)
    ↓
[Conv2D 256 + ReLU] → (16,16,256)
    ↓
[UpSample 2×2]      → (32,32,256)
    ↓
[Conv2D 3]          → (32,32,3)
    ↓
OUTPUT (32,32,3)
```

### Layer Specifications

| Layer | Type | Kernel | Output Shape | Parameters |
|:------|:-----|:-------|:-------------|:-----------|
| Input | - | - | (32, 32, 3) | 0 |
| Conv1 | Conv2D + ReLU | 3×3, pad=1 | (32, 32, 256) | 7,168 |
| Pool1 | MaxPool2D | 2×2 | (16, 16, 256) | 0 |
| Conv2 | Conv2D + ReLU | 3×3, pad=1 | (16, 16, 128) | 295,040 |
| Pool2 | MaxPool2D | 2×2 | **(8, 8, 128)** | 0 |
| Conv3 | Conv2D + ReLU | 3×3, pad=1 | (8, 8, 128) | 147,584 |
| Up1 | UpSample | 2×2 | (16, 16, 128) | 0 |
| Conv4 | Conv2D + ReLU | 3×3, pad=1 | (16, 16, 256) | 295,168 |
| Up2 | UpSample | 2×2 | (32, 32, 256) | 0 |
| Conv5 | Conv2D | 3×3, pad=1 | (32, 32, 3) | 6,915 |
| **Total** | | | | **751,875** |

### Latent Representation
- **Shape:** (8, 8, 128)
- **Dimensions:** 8 × 8 × 128 = **8,192 features**
- **Compression ratio:** 3,072 (input) → 8,192 (latent) → expansion for richer representation
- **Purpose:** Captures essential visual patterns (edges, textures, shapes)

### Training Objective
$$\mathcal{L}_{MSE} = \frac{1}{N} \sum_{i=1}^{N} ||x_i - \hat{x}_i||^2$$

where $x_i$ is the input image and $\hat{x}_i$ is the reconstructed output.

---
## 1.4 Project Objectives

### Performance Goals
| Metric | Target |
|:-------|:-------|
| Autoencoder Training Time | < 10 minutes (50K images, 20 epochs) |
| Feature Extraction Time | < 20 seconds (60K images) |
| GPU Speedup over CPU | > 20× |
| Classification Accuracy | 60-65% |

### Technical Learning Objectives
1. **CUDA Parallel Programming:**
   - Implement CNN layers as CUDA kernels (Conv2D, ReLU, MaxPool, Upsample)
   - Optimize memory access patterns (shared memory, coalescing)
   - Apply advanced techniques (tiling, kernel fusion, cuDNN)

2. **Deep Learning Implementation:**
   - Understand autoencoder architecture and training
   - Implement backpropagation from scratch
   - Handle weight initialization (He initialization)

3. **Two-Stage ML Pipeline:**
   - Unsupervised feature learning (Autoencoder)
   - Supervised classification (SVM with LIBSVM)

### Success Criteria
- ✓ Complete end-to-end pipeline: Load → Train → Extract → Classify → Evaluate
- ✓ GPU outputs match CPU within floating-point tolerance
- ✓ Achieve >20× speedup over CPU baseline
- ✓ Classification accuracy >60% on CIFAR-10 test set
- ✓ Document optimization analysis with profiling data

---
# Section 2: Implementation Phases
---

---
## Phase 2.1: CPU Baseline Implementation

### Objectives
- Set up project infrastructure and create working CPU baseline
- Implement data pipeline for CIFAR-10
- Implement all neural network layers on CPU
- Establish baseline performance metrics for comparison

### Implementation Details

#### Data Pipeline
```cpp
class CIFAR10Dataset {
    void load_batch(const std::string& path);  // Read binary files
    void normalize();                           // uint8 [0,255] → float [0,1]
    void shuffle();                             // Random shuffle per epoch
    Batch get_batch(int batch_size);           // Return mini-batch
};
```

#### Layer Implementations
- **Conv2D:** 7 nested loops (batch, out_c, h, w, in_c, kh, kw) with OpenMP
- **ReLU:** Element-wise `max(0, x)`
- **MaxPool2D:** 2×2 window max with index tracking for backprop
- **Upsample2D:** Nearest neighbor interpolation
- **MSE Loss:** Mean squared error with gradient computation

#### Training Loop Structure
```cpp
for (int epoch = 0; epoch < 20; epoch++) {
    shuffle(indices);
    for (int batch = 0; batch < num_batches; batch++) {
        // Forward pass
        Tensor output = autoencoder.forward(batch_data);
        float loss = mse_loss(output, batch_data);
        
        // Backward pass + SGD update
        Tensor grad = mse_loss_backward(output, batch_data);
        autoencoder.backward(grad, learning_rate);
    }
}
```

### Results
**Note:** CPU was run on 100 images due to extremely long training time. Results are scaled to 50,000 images (×500) for fair comparison.

In [ ]:
# Phase 1 CPU Baseline - Load pre-computed results
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

# Data from results/phase-1/cpu_phase1_log.csv (100 images)
cpu_data = """epoch,loss,time_sec
1,0.274751,30.1464
2,0.276583,30.05
3,0.280156,30.382
4,0.270955,30.5524
5,0.263449,30.5193
6,0.283664,30.5303
7,0.282206,30.5369
8,0.266745,37.4728
9,0.27516,34.5667
10,0.28457,30.6306
11,0.254605,30.5368
12,0.270007,30.5552
13,0.290835,36.53
14,0.271196,30.5217
15,0.271217,30.4278
16,0.274831,29.7033
17,0.268553,29.8357
18,0.251887,29.8761
19,0.254904,30.3387
20,0.264596,30.8022"""

df_cpu = pd.read_csv(StringIO(cpu_data))
SCALE_FACTOR_CPU = 500  # 100 -> 50,000 images
df_cpu['time_sec_scaled'] = df_cpu['time_sec'] * SCALE_FACTOR_CPU

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df_cpu['epoch'], df_cpu['loss'], 'b-o', linewidth=2, markersize=5)
axes[0].set_title('Phase 1 CPU - Training Loss', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (MSE)'); axes[0].grid(True, alpha=0.3)

axes[1].plot(df_cpu['epoch'], df_cpu['time_sec_scaled'] / 60, 'g-o', linewidth=2, markersize=5)
axes[1].set_title('Phase 1 CPU - Time/Epoch (Scaled to 50K)', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Time (minutes)'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("="*60)
print("PHASE 1 CPU BASELINE RESULTS")
print("="*60)
print(f"Training images: 100 (scaled to 50,000)")
print(f"Epochs: 20, Batch: 32, LR: 0.001")
print("-"*60)
print(f"Best Loss: {df_cpu['loss'].min():.6f}")
print(f"Final Loss: {df_cpu['loss'].iloc[-1]:.6f}")
print(f"Avg Time/Epoch (100 imgs): {df_cpu['time_sec'].mean():.2f}s")
print(f"Avg Time/Epoch (scaled 50K): {df_cpu['time_sec_scaled'].mean()/60:.1f} min")
print(f"Total Training (scaled 50K): {df_cpu['time_sec_scaled'].sum()/3600:.1f} hours")
print("="*60)

### Key Takeaways - Phase 1

#### What we learned about the algorithm:
- **Convolution is the bottleneck:** 7 nested loops with O(N × C_out × H × W × C_in × K²) complexity
- **Memory access patterns are inefficient:** High CPU cache miss rate due to non-contiguous tensor access
- **OpenMP helps but has limits:** Only utilizes physical cores (4-8), insufficient for this workload

#### Insights for GPU implementation:
- **Parallelization opportunity:** Each output pixel is independent → map 1 thread = 1 output pixel
- **Data reuse:** Same input tile read multiple times → use Shared Memory
- **Weight reuse:** Weights read repeatedly → cache in Constant/Shared Memory

---
## Phase 2.2: GPU Basic (Naive Implementation)

### Objectives
- Port layers from CPU to GPU with naive implementation
- 1 thread = 1 output element
- Global Memory for all computations

### Results (from results/phase-2/)
**Note:** Phase 2 was run on 5,000 images. Time is scaled to 50,000 images (x10).

---
## Phase 2.2: GPU Basic Implementation (Naive)

### Objectives
- Port all CPU operations to GPU with basic parallelization
- Verify correctness of GPU kernels against CPU outputs
- Establish baseline GPU performance for optimization comparison

### Implementation Details

#### Parallelization Strategy
- **1 thread = 1 output element:** Each CUDA thread computes one output pixel/element
- **2D thread blocks:** `dim3(16, 16)` = 256 threads per block for spatial operations
- **Global Memory:** All data read/written from global memory (no optimization yet)

#### Kernel Designs

**Convolution Kernel:**
```cpp
__global__ void conv2d_forward_kernel(
    const float* input, const float* weights, const float* bias,
    float* output, int batch, int in_c, int out_c, int h, int w, int k)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    // Each thread computes one output pixel
    // Nested loops over input channels and kernel
    for (int ic = 0; ic < in_c; ++ic)
        for (int kh = 0; kh < k; ++kh)
            for (int kw = 0; kw < k; ++kw)
                sum += input[...] * weights[...];
    output[idx] = sum + bias[oc];
}
```

**Pooling Kernel:**
```cpp
__global__ void maxpool2d_forward_kernel(...)
{
    // Each thread handles one 2×2 window
    // Find max value and store index for backprop
    float max_val = -FLT_MAX;
    for (int ph = 0; ph < 2; ++ph)
        for (int pw = 0; pw < 2; ++pw)
            if (input[...] > max_val) { max_val = input[...]; max_idx = ...; }
    output[idx] = max_val;
    indices[idx] = max_idx;
}
```

#### Memory Management
- `cudaMalloc` / `cudaFree` for device memory
- `cudaMallocHost` for pinned host memory (faster transfers)
- **He Initialization:** `std = sqrt(2.0 / fan_in)` using cuRAND

### Results
**Note:** Phase 2 was run on 5,000 images. Time is scaled to 50,000 images (×10) for comparison.

---
## Phase 2.3: GPU Optimized - Version 1: Tiled Convolution

### Optimization Techniques
- **Shared Memory Tiling:** Load input tiles to shared memory
- **Vectorized Memory Access:** float4 for ReLU
- **Loop Unrolling:** `#pragma unroll`

### Build and Train (50,000 images)

### Profiling Analysis - Phase 2

#### Time Distribution (Estimated)
| Operation | % of Training Time |
|:----------|:-------------------|
| Conv Forward | ~35% |
| Conv Backward (data) | ~25% |
| Conv Backward (weights) | ~20% |
| Other (ReLU, Pool, etc.) | ~15% |
| Memory Transfer | ~5% |

#### Initial Bottleneck Identification
- **Global memory bandwidth:** Each thread reads weights and input from global memory (~400-800 cycles latency)
- **No data reuse:** Same input pixels read multiple times by different threads
- **Memory access not fully coalesced:** Non-optimal access patterns reduce effective bandwidth

### Key Takeaways - Phase 2

#### What was surprisingly fast or slow?
- **Fast:** ReLU and element-wise operations are very fast due to embarrassingly parallel nature
- **Slow:** Convolution backward pass still dominates due to memory bandwidth bottleneck

#### Optimization Opportunities Identified
1. **Shared Memory Tiling:** Load input tiles to shared memory for data reuse
2. **cuDNN:** Use NVIDIA's optimized library for convolution
3. **Kernel Fusion:** Combine Conv + ReLU to reduce memory round-trips
4. **Vectorized Access:** Use float4 to increase memory throughput

In [ ]:
# Build Phase 3 - Tiled Convolution
!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -DUSE_TILED_CONV \
    -Iinclude -lcublas -lcurand \
    -o gpu_train_tiled \
    src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp

print('Build complete: Tiled Convolution')

---
## Phase 2.3: GPU Optimized Implementation - Version 1 (Tiled Convolution)

### Optimization Focus: Memory Optimization with Shared Memory Tiling

### Objectives
- Reduce global memory access by loading input tiles to shared memory
- Increase data reuse between threads in the same block
- Expected improvement: 1.5-2× over naive implementation

### Implementation Details

#### Optimization Technique: Shared Memory Tiling
**Why this should help:**
- Shared Memory latency: ~5 cycles vs Global Memory: ~400-800 cycles
- Each input pixel is used by multiple output pixels in convolution
- Loading once to shared memory eliminates redundant global reads

**Implementation Approach:**
```cpp
__global__ void conv2d_forward_tiled_kernel(...) {
    extern __shared__ float s_input[];
    
    // Cooperative loading: all threads load input tile
    for (int ic = 0; ic < in_c; ++ic) {
        // Load tile to shared memory
        int tile_h = TILE_SIZE + kernel_size - 1;
        int tile_w = TILE_SIZE + kernel_size - 1;
        
        for (int load = tid; load < tile_h * tile_w; load += blockDim.x * blockDim.y) {
            s_input[load] = input[...];  // Cooperative loading
        }
        __syncthreads();
        
        // Compute using shared memory (fast!)
        #pragma unroll
        for (int kh = 0; kh < 3; ++kh) {
            for (int kw = 0; kw < 3; ++kw) {
                sum += s_input[local_h + kh][local_w + kw] * weights[...];
            }
        }
        __syncthreads();
    }
}
```

#### Additional Optimizations Applied
| Technique | Description | Expected Speedup |
|:----------|:------------|:-----------------|
| Loop Unrolling | `#pragma unroll` for 3×3 kernel | 1.1-1.2× |
| Vectorized ReLU | `float4` processing | 1.1× |
| Pinned Memory | `cudaMallocHost` for transfers | 1.2-1.5× |
| Fast Math | `--use_fast_math` compiler flag | 1.1× |

### Build and Train (50,000 images)

In [ ]:
# Train Phase 3 - Tiled (50,000 images)
!./gpu_train_tiled --data data --epochs 20 --batch 64 --lr 0.001 \
    --log phase3_tiled.csv --log-txt phase3_tiled.txt --save-weights phase3_tiled.weights

In [ ]:
# Check GPU memory after training
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], capture_output=True, text=True)
gpu_memory_tiled = int(result.stdout.strip())
print(f"GPU Memory Used (Tiled): {gpu_memory_tiled} MB")

In [ ]:
# Visualize Phase 3 Tiled results
df_tiled = pd.read_csv('phase3_tiled.csv')
ep_tiled = df_tiled[df_tiled['batch'].isna()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep_tiled['epoch'], ep_tiled['loss'], 'r-o', linewidth=2, markersize=5)
axes[0].set_title('Phase 3 Tiled - Training Loss', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (MSE)'); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_tiled['epoch'], ep_tiled['epoch_time_sec'], 'orange', marker='o', linewidth=2, markersize=5)
axes[1].set_title('Phase 3 Tiled - Time per Epoch', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Time (seconds)'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("="*60)
print("PHASE 3 TILED CONVOLUTION RESULTS")
print("="*60)
print(f"Training images: 50,000")
print(f"Epochs: 20, Batch: 64, LR: 0.001")
print("-"*60)
print(f"Best Loss: {ep_tiled['best_loss'].iloc[-1]:.6f}")
print(f"Final Loss: {ep_tiled['loss'].iloc[-1]:.6f}")
print(f"Avg Time/Epoch: {ep_tiled['epoch_time_sec'].mean():.2f}s")
print(f"Total Training: {ep_tiled['epoch_time_sec'].sum()/60:.2f} min")
print(f"GPU Memory: {gpu_memory_tiled} MB")
print("="*60)

---
## Phase 2.4: GPU Optimized - Version 2: cuDNN

### Optimization Techniques
- **cuDNN:** NVIDIA's optimized deep learning library
- **Automatic algorithm selection** for best performance
- **Pinned Memory** for faster CPU-GPU transfers

### Build and Train (50,000 images)

### Analysis - Phase 2.3 (Tiled Convolution)

#### Why did this optimization work (or not work as expected)?
- **Overhead of shared memory management:** Loading tiles and synchronizing threads creates overhead
- **Small kernel size (3×3):** With small kernels, data reuse ratio is only ~9×, not enough to offset overhead
- **Already near bandwidth limit:** Naive implementation was already achieving reasonable bandwidth utilization

#### What did profiling reveal?
- Shared memory tiling is more effective with larger kernel sizes (5×5, 7×7)
- For 3×3 kernels, the synchronization overhead (`__syncthreads()`) is significant
- cuDNN uses more sophisticated algorithms (Winograd, FFT) for small kernels

#### What's the next bottleneck?
- Algorithm selection: Need smarter convolution algorithms
- cuDNN provides automatic algorithm selection optimized for each GPU architecture

### Key Takeaways - Phase 2.3
- **Lesson:** Not all optimizations help equally - profile before and after
- **Applicability:** Tiling is more effective for matrix multiplication and larger convolution kernels

In [ ]:
# Build Phase 3 - cuDNN
!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -Iinclude -lcublas -lcudnn \
    -o gpu_train_opt \
    src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp

print('Build complete: cuDNN')

---
## Phase 2.4: GPU Optimized Implementation - Version 2 (cuDNN)

### Optimization Focus: cuDNN Library Integration

### Objectives
- Use NVIDIA's highly optimized cuDNN library for convolution operations
- Leverage automatic algorithm selection for best performance
- Expected improvement: 2-5× over naive implementation

### Implementation Details

#### Optimization Technique: cuDNN
**Why cuDNN should help:**
- NVIDIA's production-quality deep learning primitives
- Automatic algorithm selection (IMPLICIT_GEMM, WINOGRAD, FFT, etc.)
- Optimized for each GPU architecture (Tensor Cores on Volta+)
- Highly tuned memory access patterns

**cuDNN Functions Used:**
| Function | Purpose |
|:---------|:--------|
| `cudnnConvolutionForward` | Optimized convolution forward pass |
| `cudnnConvolutionBackwardData` | Gradient w.r.t. input |
| `cudnnConvolutionBackwardFilter` | Gradient w.r.t. weights |
| `cudnnConvolutionBackwardBias` | Gradient w.r.t. bias |

**Implementation Approach:**
```cpp
// Setup descriptors
cudnnSetTensor4dDescriptor(inputDesc, CUDNN_TENSOR_NCHW, ...);
cudnnSetFilter4dDescriptor(filterDesc, CUDNN_TENSOR_NCHW, ...);
cudnnSetConvolution2dDescriptor(convDesc, pad, pad, 1, 1, ...);

// Find best algorithm
cudnnGetConvolutionForwardAlgorithm(handle, inputDesc, filterDesc, 
    convDesc, outputDesc, CUDNN_CONVOLUTION_FWD_PREFER_FASTEST, ...);

// Execute convolution
cudnnConvolutionForward(handle, &alpha, inputDesc, input,
    filterDesc, weights, convDesc, algo, workspace, workspaceSize,
    &beta, outputDesc, output);
```

#### Additional Optimizations Applied
- **Pinned Memory (#5):** `cudaMallocHost` for faster CPU-GPU transfers
- **Double Buffering:** Overlap transfer and compute with CUDA streams
- **Loop Unrolling (#10):** `#pragma unroll` in custom kernels
- **Fast Math:** `--use_fast_math` compiler flag

### Build and Train (50,000 images)

In [ ]:
# Check GPU memory
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], capture_output=True, text=True)
gpu_memory_opt = int(result.stdout.strip())
print(f"GPU Memory Used (cuDNN): {gpu_memory_opt} MB")

In [ ]:
# Visualize Phase 3 cuDNN results
df_opt = pd.read_csv('phase3_opt.csv')
ep_opt = df_opt[df_opt['batch'].isna()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep_opt['epoch'], ep_opt['loss'], 'm-o', linewidth=2, markersize=5)
axes[0].set_title('Phase 3 cuDNN - Training Loss', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (MSE)'); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_opt['epoch'], ep_opt['epoch_time_sec'], 'purple', marker='o', linewidth=2, markersize=5)
axes[1].set_title('Phase 3 cuDNN - Time per Epoch', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Time (seconds)'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("="*60)
print("PHASE 3 cuDNN RESULTS")
print("="*60)
print(f"Training images: 50,000")
print(f"Epochs: 20, Batch: 64, LR: 0.001")
print("-"*60)
print(f"Best Loss: {ep_opt['best_loss'].iloc[-1]:.6f}")
print(f"Final Loss: {ep_opt['loss'].iloc[-1]:.6f}")
print(f"Avg Time/Epoch: {ep_opt['epoch_time_sec'].mean():.2f}s")
print(f"Total Training: {ep_opt['epoch_time_sec'].sum()/60:.2f} min")
print(f"GPU Memory: {gpu_memory_opt} MB")
print("="*60)

---
## Phase 2.5: GPU Optimized - Version 3: BCE Loss

### Changes from MSE
- **Output Activation:** Sigmoid after final Conv
- **Loss Function:** Binary Cross-Entropy instead of MSE
- **Output range:** Bounded [0, 1]

### Build and Train (50,000 images)

In [ ]:
# Build Phase 3 - BCE (same binary, different runtime flag)
!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -Iinclude -lcublas -lcudnn \
    -o gpu_train_bce \
    src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp

print('Build complete: BCE Loss')

### Analysis - Phase 2.4 (cuDNN)

#### Why did this optimization work (or not work as expected)?
- **Setup overhead:** cuDNN needs time to select algorithm and allocate workspace
- **Small model size:** With only 751K parameters, cuDNN overhead is proportionally larger
- **Already optimized naive kernels:** Our naive kernels had reasonable memory coalescing

#### What did profiling reveal?
- cuDNN shines with larger models (ResNet, VGG) and larger batch sizes (128+)
- For small models, the algorithm selection and workspace allocation overhead is significant
- cuDNN is still the best choice for production due to portability and maintenance

#### What's the next bottleneck?
- Loss function: Try BCE loss with Sigmoid for bounded outputs
- Architecture: Deeper networks might learn better features

### Key Takeaways - Phase 2.4
- **Lesson:** cuDNN is best for production, custom kernels are valuable for learning
- **Applicability:** Always use cuDNN for production deep learning applications

In [ ]:
# Train Phase 3 - BCE Loss (50,000 images)
!./gpu_train_bce --data data --epochs 20 --batch 64 --lr 0.001 \
    --bce-loss \
    --log phase3_bce.csv --log-txt phase3_bce.txt --save-weights phase3_bce.weights

---
## Phase 2.5: GPU Optimized Implementation - Version 3 (BCE Loss)

### Optimization Focus: Alternative Loss Function

### Objectives
- Experiment with Binary Cross-Entropy loss instead of MSE
- Bounded output [0, 1] with Sigmoid activation
- Compare reconstruction quality and downstream classification accuracy

### Implementation Details

#### Changes from MSE Version
| Aspect | MSE (Previous) | BCE + Sigmoid (This Version) |
|:-------|:---------------|:-----------------------------|
| Output Activation | None (unbounded) | Sigmoid [0, 1] |
| Loss Function | Mean Squared Error | Binary Cross-Entropy |
| Gradient Behavior | Linear | Stronger near 0.5, weaker near 0/1 |
| Pixel Validity | May need clipping | Always valid |

#### BCE Loss Formula
$$\mathcal{L}_{BCE} = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(\hat{y}_i + \epsilon) + (1-y_i) \log(1-\hat{y}_i + \epsilon)]$$

where $\epsilon = 10^{-7}$ for numerical stability.

#### Numerically Stable Sigmoid
```cpp
__device__ float stable_sigmoid(float x) {
    if (x >= 0) return 1.0f / (1.0f + expf(-x));
    else {
        float exp_x = expf(x);
        return exp_x / (1.0f + exp_x);
    }
}
```

### Build and Train (50,000 images)

In [ ]:
# Visualize Phase 3 BCE results
df_bce = pd.read_csv('phase3_bce.csv')
ep_bce = df_bce[df_bce['batch'].isna()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep_bce['epoch'], ep_bce['loss'], 'c-o', linewidth=2, markersize=5)
axes[0].set_title('Phase 3 BCE - Training Loss', fontsize=11)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (BCE)'); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_bce['epoch'], ep_bce['epoch_time_sec'], 'teal', marker='o', linewidth=2, markersize=5)
axes[1].set_title('Phase 3 BCE - Time per Epoch', fontsize=11)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Time (seconds)'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("="*60)
print("PHASE 3 BCE LOSS RESULTS")
print("="*60)
print(f"Training images: 50,000")
print(f"Epochs: 20, Batch: 64, LR: 0.001")
print("-"*60)
print(f"Best Loss: {ep_bce['best_loss'].iloc[-1]:.6f}")
print(f"Final Loss: {ep_bce['loss'].iloc[-1]:.6f}")
print(f"Avg Time/Epoch: {ep_bce['epoch_time_sec'].mean():.2f}s")
print(f"Total Training: {ep_bce['epoch_time_sec'].sum()/60:.2f} min")
print(f"GPU Memory: {gpu_memory_bce} MB")
print("="*60)

---
# Section 3: Comprehensive Performance Analysis
---

## 3.1 Performance Comparison Across All Phases

### Performance Summary Table

| Phase | Training Time | Speedup (vs CPU) | Incremental Speedup | Memory Usage | Key Optimization |
|:------|:--------------|:-----------------|:--------------------|:-------------|:-----------------|
| CPU Baseline | ~87 hours* | 1.0× | - | ~500 MB | - |
| GPU Basic | ~102 min* | ~51× | 51× | ~1.5 GB | Parallelization |
| GPU Tiled | ~10 min | ~509× | ~10× | ~2.3 GB | Shared Memory |
| GPU cuDNN | ~11 min | ~475× | ~0.9× | ~2.5 GB | cuDNN Library |
| GPU BCE | ~11 min | ~475× | ~1.0× | ~2.5 GB | BCE Loss |

*Scaled to 50,000 images for fair comparison

### Visualization

In [ ]:
# Comprehensive Performance Comparison
import matplotlib.pyplot as plt
import numpy as np

# Collect all results
phases = ['CPU\n(scaled)', 'GPU Naive\n(scaled)', 'GPU Tiled', 'GPU cuDNN', 'GPU BCE']

# Times (scaled to 50K where applicable)
total_times = [
    df_cpu['time_sec_scaled'].sum(),  # CPU scaled
    df_phase2['time_sec_scaled'].sum(),  # Phase 2 scaled
    ep_tiled['epoch_time_sec'].sum(),  # Phase 3 Tiled
    ep_opt['epoch_time_sec'].sum(),  # Phase 3 cuDNN
    ep_bce['epoch_time_sec'].sum()  # Phase 3 BCE
]

final_losses = [
    df_cpu['loss'].iloc[-1],
    df_phase2['loss'].iloc[-1],
    ep_tiled['loss'].iloc[-1],
    ep_opt['loss'].iloc[-1],
    ep_bce['loss'].iloc[-1]
]

# Calculate speedups vs CPU
cpu_time = total_times[0]
speedups = [cpu_time / t for t in total_times]

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#ff6b6b', '#feca57', '#48dbfb', '#1dd1a1', '#a55eea']

# Training Time
bars = axes[0].bar(phases, [t/60 for t in total_times], color=colors, edgecolor='black')
axes[0].set_ylabel('Total Training Time (minutes)')
axes[0].set_title('Training Time Comparison')
axes[0].set_yscale('log')
for bar, t in zip(bars, total_times):
    label = f'{t/3600:.1f}h' if t >= 3600 else f'{t/60:.1f}m'
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.1, label, ha='center', fontsize=9)

# Speedup
axes[1].bar(phases, speedups, color=colors, edgecolor='black')
axes[1].set_ylabel('Speedup (vs CPU)')
axes[1].set_title('Speedup Comparison')
for i, sp in enumerate(speedups):
    axes[1].text(i, sp + max(speedups)*0.02, f'{sp:.0f}x', ha='center', fontsize=9)

# Final Loss
axes[2].bar(phases, final_losses, color=colors, edgecolor='black')
axes[2].set_ylabel('Final Loss')
axes[2].set_title('Final Loss Comparison')
for i, loss in enumerate(final_losses):
    axes[2].text(i, loss + max(final_losses)*0.02, f'{loss:.4f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=150)
plt.show()

# Summary Table
print("="*80)
print("PERFORMANCE SUMMARY")
print("="*80)
print(f"{'Phase':<20} {'Total Time':<15} {'Speedup':<12} {'Final Loss':<12} {'Memory'}")
print("-"*80)
memories = ['~500 MB (CPU)', '~1.5 GB', f'{gpu_memory_tiled} MB', f'{gpu_memory_opt} MB', f'{gpu_memory_bce} MB']
for i, phase in enumerate(['CPU (scaled)', 'GPU Naive (scaled)', 'GPU Tiled', 'GPU cuDNN', 'GPU BCE']):
    time_str = f"{total_times[i]/3600:.1f}h" if total_times[i] >= 3600 else f"{total_times[i]/60:.1f}min"
    print(f"{phase:<20} {time_str:<15} {speedups[i]:<12.0f}x {final_losses[i]:<12.4f} {memories[i]}")
print("="*80)

---
# Section 4: SVM Classification (Phase 4)
---

Compare SVM accuracy using features from different autoencoder weights:
- Phase 2 weights (GPU Naive)
- Phase 3 Tiled weights
- Phase 3 cuDNN weights
- Phase 3 BCE weights

### Analysis - Phase 2.5 (BCE Loss)

#### Why did this optimization work (or not work as expected)?
- **Different loss scale:** BCE loss values are higher than MSE (different scale, not directly comparable)
- **Bounded outputs:** Sigmoid ensures valid pixel values [0, 1] without clipping
- **Similar training time:** BCE computation is similar complexity to MSE

#### Comparison: MSE vs BCE
| Metric | MSE Loss | BCE Loss |
|:-------|:---------|:---------|
| Final Loss | ~0.011 | ~0.58 |
| Training Time | ~10 min | ~10 min |
| Output Range | Unbounded | [0, 1] |

#### Key Takeaways - Phase 2.5
- **Lesson:** Loss function choice depends on the problem - MSE works well for reconstruction
- **Applicability:** BCE is better when probabilistic interpretation is needed

In [ ]:
# Build feature extractor
!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    -DUSE_OPTIMIZED_KERNELS -Iinclude -lcublas -lcudnn \
    -o extract_features \
    src/extract_features.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp

print('Build complete: Feature Extractor')

---
## Phase 2.6: SVM Integration

### Objectives
- Extract features using trained encoder
- Train SVM classifier on learned features
- Evaluate end-to-end classification performance

### Implementation Details

#### Feature Extraction
- Load trained encoder weights
- Run encoder forward pass only (no decoder)
- Output: 8,192-dim feature vector per image
- Extract features for all 60,000 images (50K train + 10K test)

#### Feature Preprocessing
```python
# StandardScaler: zero mean, unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA: reduce dimensionality 8192 -> 640
pca = PCA(n_components=640, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
```

#### SVM Configuration (LIBSVM via scikit-learn)
| Parameter | Value | Reason |
|:----------|:------|:-------|
| Kernel | RBF | Non-linear decision boundary |
| C | 10 | Regularization strength |
| gamma | 'scale' | 1 / (n_features × variance) |

### Build Feature Extractor

In [ ]:
# Evaluate all weights
results = {}

# Phase 3 Tiled
acc, t = evaluate_weights('phase3_tiled.weights', 'tiled')
results['Phase 3 Tiled'] = {'accuracy': acc, 'svm_time': t}

# Phase 3 cuDNN
acc, t = evaluate_weights('phase3_opt.weights', 'cudnn')
results['Phase 3 cuDNN'] = {'accuracy': acc, 'svm_time': t}

# Phase 3 BCE
acc, t = evaluate_weights('phase3_bce.weights', 'bce')
results['Phase 3 BCE'] = {'accuracy': acc, 'svm_time': t}

In [ ]:
# Summary of SVM results
print("\n" + "="*60)
print("SVM CLASSIFICATION RESULTS COMPARISON")
print("="*60)
print(f"{'Weights Source':<25} {'Test Accuracy':<15} {'SVM Train Time'}")
print("-"*60)
for name, data in results.items():
    print(f"{name:<25} {data['accuracy']*100:.2f}%{'':<8} {data['svm_time']:.2f}s")
print("="*60)

# Plot accuracy comparison
plt.figure(figsize=(10, 5))
names = list(results.keys())
accuracies = [results[n]['accuracy']*100 for n in names]
colors = ['#48dbfb', '#1dd1a1', '#a55eea']

bars = plt.bar(names, accuracies, color=colors, edgecolor='black')
plt.ylabel('Test Accuracy (%)')
plt.title('SVM Classification Accuracy by Autoencoder Weights')
plt.ylim([0, 100])

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{acc:.1f}%', ha='center', fontsize=11)

plt.axhline(y=60, color='r', linestyle='--', label='Target: 60%')
plt.legend()
plt.tight_layout()
plt.savefig('svm_accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
# Function to extract features and train SVM with full evaluation
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

def evaluate_weights_full(weights_path, name, show_confusion=True):
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")
    
    # Extract features
    print("Extracting features...")
    start_extract = time.time()
    !./extract_features --data data --weights {weights_path} --output features_{name}
    extract_time = time.time() - start_extract
    print(f"Feature extraction time: {extract_time:.2f}s")
    
    # Load features
    X_train = np.fromfile(f'features_{name}_train.bin', dtype=np.float32).reshape(-1, 8192)
    X_test = np.fromfile(f'features_{name}_test.bin', dtype=np.float32).reshape(-1, 8192)
    y_train = train_labels
    y_test = test_labels
    
    print(f"Train features: {X_train.shape}")
    print(f"Test features: {X_test.shape}")
    
    # Preprocessing
    print("Preprocessing (StandardScaler + PCA)...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    pca = PCA(n_components=640, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    print(f"After PCA: {X_train_pca.shape}")
    
    # Train SVM
    print("Training SVM (RBF kernel, C=10)...")
    start_svm = time.time()
    svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
    svm.fit(X_train_pca, y_train)
    svm_time = time.time() - start_svm
    print(f"SVM training time: {svm_time:.2f}s")
    
    # Evaluate
    y_pred = svm.predict(X_test_pca)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\nTest Accuracy: {accuracy*100:.2f}%")
    
    # Per-class accuracy
    print("\n" + "="*60)
    print("PER-CLASS ACCURACY")
    print("="*60)
    cm = confusion_matrix(y_test, y_pred)
    for i, label in enumerate(CIFAR10_LABELS):
        class_acc = cm[i, i] / cm[i].sum() * 100
        print(f"{label:12s}: {class_acc:.1f}%")
    
    # Confusion Matrix
    if show_confusion:
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=CIFAR10_LABELS, yticklabels=CIFAR10_LABELS)
        plt.title(f'Confusion Matrix - {name}')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(f'confusion_matrix_{name}.png', dpi=150)
        plt.show()
    
    return {
        'accuracy': accuracy,
        'svm_time': svm_time,
        'extract_time': extract_time,
        'confusion_matrix': cm
    }

---
# Section 4: Lessons Learned and Challenges
---

## 4.1 Key Technical Insights

### CUDA Programming
- **Memory Hierarchy Matters:** Global memory (~400-800 cycles) vs Shared memory (~5 cycles) vs Registers (~1 cycle)
- **Coalesced Access:** Threads in same warp should access contiguous memory for maximum bandwidth
- **Occupancy vs Registers:** More registers per thread → fewer threads per SM → lower occupancy

### Deep Learning
- **Weight Initialization is Critical:** Zero initialization causes symmetry problem; He initialization (`std = sqrt(2/fan_in)`) works well with ReLU
- **Loss Function Choice:** MSE for regression/reconstruction, BCE for probabilistic interpretation
- **Feature Quality:** Autoencoder features capture reconstruction info, not necessarily discriminative info

### Performance Optimization
- **Profile Before Optimize:** Not all optimizations help - measure before and after
- **cuDNN for Production:** NVIDIA's library is highly optimized for each GPU architecture
- **Custom Kernels for Learning:** Writing kernels from scratch provides deep understanding

## 4.2 Major Challenges and Solutions

### Challenge 1: Weight Initialization Bug
- **Problem:** Weights initialized to zeros caused model to output constant gray images (mean value)
- **Solution:** Implemented He initialization using cuRAND: `std = sqrt(2.0 / fan_in)`
- **Lesson:** Always verify weight statistics after initialization (mean ≈ 0, std ≈ expected)

### Challenge 2: CUDA Memory Management
- **Problem:** Memory leaks and invalid memory access errors during training
- **Solution:** Careful `cudaMalloc`/`cudaFree` pairing, `cudaDeviceSynchronize()` before free
- **Lesson:** GPU memory management requires same discipline as C/C++ - use RAII patterns

### Challenge 3: Numerical Stability
- **Problem:** NaN/Inf values appearing during BCE loss computation
- **Solution:** Numerically stable sigmoid implementation, epsilon in log operations
- **Lesson:** Always check for NaN/Inf in debug builds, use stable math formulations

---
# Section 5: Conclusion and Future Work
---

## 5.1 Project Summary

### Final Performance Metrics

| Metric | Target | Achieved | Status |
|:-------|:-------|:---------|:-------|
| Training Time (50K images) | < 10 min | ~10 min | ✓ Achieved |
| Feature Extraction (60K) | < 20 sec | ~40 sec | ⚠ Partial |
| GPU Speedup vs CPU | > 20× | ~500× | ✓✓ Exceeded |
| Classification Accuracy | 60-65% | ~48-52% | ⚠ Below target |

### Pipeline Completion
- ✓ **Data Loading:** CIFAR-10 binary format parsing, normalization [0,1]
- ✓ **Autoencoder Training:** GPU-accelerated with multiple optimization levels
- ✓ **Feature Extraction:** 8,192-dim latent vectors from encoder
- ✓ **SVM Classification:** RBF kernel with PCA preprocessing
- ✓ **Evaluation:** Accuracy, confusion matrix, per-class analysis

## 5.2 Key Achievements

### Technical Achievements
1. **Complete CUDA Implementation:** All neural network layers implemented from scratch (Conv2D, ReLU, MaxPool, Upsample, MSE/BCE Loss)
2. **Multiple Optimization Levels:** Naive → Tiled → cuDNN progression with analysis
3. **End-to-End Pipeline:** From raw images to classification predictions

### Performance Achievements
- **~500× speedup** over CPU baseline (scaled comparison)
- **~10 minutes** training time for 50K images × 20 epochs
- **0.011 MSE loss** - model learns good reconstruction

## 5.3 Limitations

### Current Bottlenecks
1. **Accuracy Gap:** Target 60-65%, achieved ~48-52%
   - Shallow architecture (only 2 conv layers in encoder)
   - Autoencoder learns reconstruction, not discrimination

2. **Feature Extraction Time:** Target <20s, achieved ~40s
   - Memory transfer overhead between CPU-GPU

3. **Architecture Constraints:**
   - No skip connections → information loss through bottleneck
   - No batch normalization → potential training instability

## 5.4 Future Improvements

### Short-term
- Deeper encoder (4-6 conv layers) for richer features
- Batch normalization for stable training
- Data augmentation (flip, crop, color jitter)

### Long-term
- Variational Autoencoder (VAE) for better latent space
- Contrastive learning instead of reconstruction
- Neural network classifier instead of SVM
- Multi-GPU training for larger models

In [ ]:
# Download all results
from google.colab import files

# Training logs
files.download('phase3_tiled.csv')
files.download('phase3_opt.csv')
files.download('phase3_bce.csv')

# Weights
files.download('phase3_tiled.weights')
files.download('phase3_opt.weights')
files.download('phase3_bce.weights')

# Plots
files.download('performance_comparison.png')
files.download('svm_accuracy_comparison.png')

print("All files downloaded!")